# HW3: Balancing with LQR

**NTUT MuJoCo Course - Lecture 4**

In lecture 3 you found a configuration where pineapple_v0 stands in perfect static balance.
That equilibrium is also perfectly **unstable**: nudge the robot and it falls over. This
homework builds the controller that keeps it upright.

The plan is the one that runs through most of modern control:

1. **Reduce** the 12-DOF robot to a 4-state cart-pole, by freezing the legs at the equilibrium.
2. **Linearize** that model about the upright equilibrium to get $\dot x = Ax + Bu$.
3. **Solve** the Riccati equation for the optimal feedback $u = -Kx$, and simulate it.

| Part | Points | Topic |
| --- | --- | --- |
| A | 8 | Reduce the robot to a cart-pole |
| B | 7 | Linearize and discretize |
| C | 10 | Riccati, LQR gain, closed-loop simulation |
| D | +5 bonus | Tuning Q and R, and what the model leaves out |

### Rules

- **DO NOT USE `while` LOOPS** anywhere in this homework.
- **DO NOT ALTER THE TEST CELLS.** They are marked `TEST CELL`, and we check.
- Before submitting: **Kernel -> Restart Kernel and Run All Cells**.

In [ ]:
%matplotlib inline
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
from dataclasses import dataclass

# pineapple_model.py is given code and lives next to this notebook's lecture folder.
import sys
from pathlib import Path
for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / 'lecture4' / 'pineapple_model.py').exists():
        sys.path.insert(0, str(_root / 'lecture4'))
        break
else:
    raise FileNotFoundError(
        'Could not locate lecture4/pineapple_model.py. Run this notebook from inside '
        'its own lecture folder in the course repository.')

from pineapple_model import (PineappleModel, WHEEL_BODIES, solve_equilibrium, vec)

np.set_printoptions(precision=6, suppress=True)
print('imports ok')

## Part A (8 pts): Cart-pole System

Balancing on two wheels is the cart-pole problem wearing a different hat. The wheels are
the cart; the body is the pole; the wheel motors push the cart. Holding the thigh and calf
joints at the lecture 3 equilibrium freezes them, so in the sagittal plane only two degrees
of freedom survive:

- $s$ - how far the contact point has rolled along the ground (m)
- $\theta$ - body pitch away from upright (rad)

State $x = [s, \theta, \dot s, \dot\theta]$, and the input $u$ is the **total** wheel torque
in Nm, delivered as $\tau_L = \tau_R = u/2$.

### Collecting the constants

Six numbers describe the reduced model. Five come straight out of the MJCF; the sixth,
the body's pitch inertia, needs the parallel axis theorem, and
`model.composite_inertia(q, exclude=WHEEL_BODIES)` is given to you for exactly that.

| symbol | meaning |
| --- | --- |
| $M$ | sprung mass - the whole robot except the wheels |
| $\ell$ | distance from the wheel axle to the sprung centre of mass |
| $I_b$ | sprung pitch inertia about that centre of mass |
| $m_w$, $I_w$ | mass and spin inertia of both wheels |
| $r$ | wheel radius |

From these, three combinations do all the work:

$$a = M + m_w + \frac{I_w}{r^2}, \qquad b = M\ell, \qquad c = I_b + M\ell^2$$

$a$ is how much inertia you feel pushing the robot along the ground (the wheels must spin
up too). $c$ is the pitch inertia about the axle. $b$ couples them: accelerating the cart
tips the pole, which is the whole difficulty of the problem.

In [ ]:
# this cell is given, no TODOs here
TARGET_HEIGHT = 0.30
model = PineappleModel()
equilibrium = solve_equilibrium(model, TARGET_HEIGHT)
q_star = equilibrium['q']
print(f'equilibrium found in {equilibrium["iterations"]} iterations at '
      f'{TARGET_HEIGHT:.2f} m; joint angles {np.round(q_star[6:], 6)}')

sprung_mass, sprung_com, sprung_inertia = model.composite_inertia(q_star, exclude=WHEEL_BODIES)
axle = np.asarray(model.evaluate(q_star)[4]).mean(axis=1)
wheel_mass = sum(model.inertials[name]['mass'] for name in WHEEL_BODIES)
# diaginertia is along each body's principal axes; the wheel's spin axis is the largest.
wheel_inertia = sum(max(model.inertials[name]['diaginertia']) for name in WHEEL_BODIES)

print(f'sprung mass   M   = {sprung_mass:.6f} kg')
print(f'sprung com        = {np.round(sprung_com, 6)}   axle = {np.round(axle, 6)}')
print(f'pitch inertia I_b = {sprung_inertia[1, 1]:.6e} kg m^2')
print(f'wheels        m_w = {wheel_mass:.4f} kg, I_w = {wheel_inertia:.6e} kg m^2')
print(f'wheel radius  r   = {model.radii[0]} m')

In [ ]:
@dataclass(frozen=True)
class PendulumParameters:
    M: float        # sprung mass (kg)
    l: float        # axle to sprung center of mass (m)
    I_b: float      # sprung pitch inertia about that center (kg m^2)
    m_w: float      # mass of both wheels (kg)
    I_w: float      # spin inertia of both wheels (kg m^2)
    r: float        # wheel radius (m)
    g: float        # gravity (m/s^2)

    @property
    def a(self):
        # TODO: effective translational inertia, including spinning the wheels up
        raise NotImplementedError('parameter a not implemented')

    @property
    def b(self):
        # TODO: translation-rotation coupling
        raise NotImplementedError('parameter b not implemented')

    @property
    def c(self):
        # TODO: pitch inertia about the AXLE, not about the center of mass
        raise NotImplementedError('parameter c not implemented')


params = PendulumParameters(
    M=float(sprung_mass),
    l=float(np.linalg.norm(sprung_com - axle)),
    I_b=float(sprung_inertia[1, 1]),
    m_w=float(wheel_mass),
    I_w=float(wheel_inertia),
    r=float(model.radii[0]),
    g=float(-model.gravity[2]))
print(f'l = {params.l:.6f} m')

### The equations of motion

Lagrange's equations for this system give

$$
\begin{bmatrix} a & b\cos\theta \\ b\cos\theta & c \end{bmatrix}
\begin{bmatrix} \ddot s \\ \ddot\theta \end{bmatrix}
=
\begin{bmatrix} b\,\dot\theta^2 \sin\theta + u/r \\ Mg\ell\sin\theta - u \end{bmatrix}
$$

Read the right-hand side physically. The wheel torque $u$ produces a ground force $u/r$ on
the cart and an equal and opposite reaction $-u$ on the body - push the cart forward and the
body tips backward. $Mg\ell\sin\theta$ is gravity *destabilizing* the pole: tip a little and
gravity tips you further. $b\dot\theta^2\sin\theta$ is the centrifugal term from the body
swinging about the axle.

Note $\theta = 0$ is **upright**, so $\cos\theta = 1$ there and the mass matrix is at its
most coupled.

In [ ]:
def mass_matrix(params, theta):
    """The configuration-dependent 2x2 mass matrix above."""
    # TODO: build it with ca.vertcat(ca.horzcat(...), ca.horzcat(...))
    raise NotImplementedError('mass matrix not implemented')


def pendulum_dynamics(params, x, u):
    """xdot = pendulum_dynamics(params, x, u) for x = [s, theta, s_dot, theta_dot]."""
    theta, s_dot, theta_dot = x[1], x[2], x[3]
    # TODO: build the right-hand side of the equations of motion (a 2-vector)
    # TODO: solve for the accelerations - use ca.solve(mass_matrix(...), rhs),
    #       NOT an explicit inverse
    # TODO: return ca.vertcat(s_dot, theta_dot, s_ddot, theta_ddot)
    raise NotImplementedError('pendulum dynamics not implemented')


def pendulum_energy(params, x):
    """Kinetic + potential energy, potential measured from the axle height."""
    theta, s_dot, theta_dot = x[1], x[2], x[3]
    kinetic = (0.5 * params.a * s_dot**2
               + params.b * s_dot * theta_dot * ca.cos(theta)
               + 0.5 * params.c * theta_dot**2)          # given
    # TODO: add the potential energy of the sprung mass
    raise NotImplementedError('pendulum energy not implemented')

In [ ]:
# ------------------------------------------------------------------- TEST CELL (A)
M2 = np.asarray(ca.DM(mass_matrix(params, 0.0)))
assert M2.shape == (2, 2)
assert np.allclose(M2, M2.T), 'the mass matrix must be symmetric'
assert np.all(np.linalg.eigvalsh(M2) > 0), 'the mass matrix must be positive definite'
np.testing.assert_allclose([M2[0, 0], M2[0, 1], M2[1, 1]],
                           [params.a, params.b, params.c], rtol=1e-12)

# Standing straight up and hanging straight down are both equilibria of the free system.
for theta in (0.0, np.pi):
    f = vec(pendulum_dynamics(params, np.array([0.0, theta, 0.0, 0.0]), 0.0))
    np.testing.assert_allclose(f, 0, atol=1e-12)

# Tipped forward with the motors off, it keeps tipping forward - that is the instability.
assert float(vec(pendulum_dynamics(params, np.array([0, 0.05, 0, 0]), 0.0))[3]) > 0
# Pushing the cart forward tips the body backward.
assert float(vec(pendulum_dynamics(params, np.zeros(4), 1.0))[3]) < 0

# With no input the system is conservative: dE/dt must vanish along the flow.
xs = ca.SX.sym('x', 4)
rate = ca.Function('rate', [xs], [ca.jacobian(pendulum_energy(params, xs), xs)
                                  @ pendulum_dynamics(params, xs, 0.0)])
for state in ([0.1, 0.2, 0.3, 0.4], [-0.5, 1.1, 0.2, -0.7]):
    np.testing.assert_allclose(float(rate(state)), 0, atol=1e-10)

print(f'a = {params.a:.6f}   b = {params.b:.6f}   c = {params.c:.6f}')
print('Part A tests passed.')

## Part B (7 pts): Linearize and discretize

LQR needs a **linear** model. Close to upright, $\sin\theta \approx \theta$ and
$\cos\theta \approx 1$, and the quadratic velocity term vanishes. Rather than doing that
expansion by hand, let CasADi differentiate the dynamics you just wrote:

$$A = \left.\frac{\partial f}{\partial x}\right|_{x^*, u^*}, \qquad
  B = \left.\frac{\partial f}{\partial u}\right|_{x^*, u^*}$$

about the upright equilibrium $x^* = 0$, $u^* = 0$.

Our controller runs at a fixed rate, so we also need the **discrete** model
$x_{k+1} = A_d x_k + B_d u_k$. Get it by differentiating one RK4 step of the nonlinear
dynamics - the same RK4 you wrote in lecture 2, now with the input held constant across
the step (a zero-order hold, which is what a real controller does).

In [ ]:
EQUILIBRIUM_STATE = np.zeros(4)
EQUILIBRIUM_INPUT = 0.0
DT = 0.01


def rk4_step(params, x, u, dt):
    """One classical RK4 step, holding u constant across the interval."""
    # TODO: the lecture 2 integrator, now carrying an input argument
    raise NotImplementedError('rk4 step not implemented')


def linearize(params):
    """Continuous-time A (4x4) and B (4x1) about the upright equilibrium."""
    x, u = ca.SX.sym('x', 4), ca.SX.sym('u')
    # TODO: build f = pendulum_dynamics(params, x, u), take ca.jacobian of it wrt
    #       x and u, wrap in a ca.Function and evaluate at the equilibrium.
    raise NotImplementedError('linearize not implemented')


def discretize(params, dt):
    """Discrete-time A_d, B_d from the RK4 map."""
    x, u = ca.SX.sym('x', 4), ca.SX.sym('u')
    # TODO: same idea, but differentiate rk4_step(params, x, u, dt) instead
    raise NotImplementedError('discretize not implemented')


def controllability(A, B):
    """[B, AB, A^2 B, ...] - full rank means every state can be steered."""
    # TODO: build it with np.linalg.matrix_power and np.hstack
    raise NotImplementedError('controllability not implemented')

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B)
A, B = linearize(params)
Ad, Bd = discretize(params, DT)
assert A.shape == (4, 4) and B.shape == (4, 1)
assert Ad.shape == (4, 4) and Bd.shape == (4, 1)

# The top rows just say that the derivative of position is velocity.
np.testing.assert_allclose(A[:2], [[0, 0, 1, 0], [0, 0, 0, 1]], atol=1e-12)
np.testing.assert_allclose(B[:2], 0, atol=1e-12)

# Symbolic derivatives must agree with finite differences.
eps = 1e-6
for column in range(4):
    e = np.eye(4)[column]
    fd = (vec(pendulum_dynamics(params, EQUILIBRIUM_STATE + eps * e, 0.0))
          - vec(pendulum_dynamics(params, EQUILIBRIUM_STATE - eps * e, 0.0))) / (2 * eps)
    np.testing.assert_allclose(A[:, column], fd, atol=1e-6)
fd = (vec(pendulum_dynamics(params, EQUILIBRIUM_STATE, eps))
      - vec(pendulum_dynamics(params, EQUILIBRIUM_STATE, -eps))) / (2 * eps)
np.testing.assert_allclose(B.flatten(), fd, atol=1e-6)

# The upright equilibrium must be unstable, at the analytic growth rate.
eigenvalues = np.linalg.eigvals(A)
growth = np.max(eigenvalues.real)
expected = np.sqrt(params.M * params.g * params.l * params.a
                   / (params.a * params.c - params.b**2))
assert growth > 1.0, 'upright should be unstable'
np.testing.assert_allclose(growth, expected, rtol=1e-9)

# And it must be controllable, or no feedback can save it.
assert np.linalg.matrix_rank(controllability(A, B)) == 4, 'continuous model uncontrollable'
assert np.linalg.matrix_rank(controllability(Ad, Bd)) == 4, 'discrete model uncontrollable'

# One RK4 step should match the exact matrix exponential to 5th order.
from scipy.linalg import expm
block = np.zeros((5, 5))
block[:4, :4], block[:4, 4:] = A, B
exact = expm(block * DT)
np.testing.assert_allclose(Ad, exact[:4, :4], atol=1e-5, rtol=1e-3)
np.testing.assert_allclose(Bd, exact[:4, 4:], atol=1e-5, rtol=1e-3)

print(f'continuous eigenvalues: {np.round(eigenvalues, 4)}')
print(f'unstable pole {growth:.4f} rad/s -> the robot falls with a {1 / growth:.3f} s time constant')
print(f'discrete eigenvalues:   {np.round(np.linalg.eigvals(Ad), 6)}')
print('\nPart B tests passed.')

That unstable pole is the whole problem in one number: left alone, any tilt grows by a
factor of $e$ every $1/\lambda$ seconds. The controller has to beat that.

## Part C (10 pts): LQR

We want the input sequence minimizing

$$J = \sum_{k=0}^{\infty} x_k^T Q x_k + u_k^T R u_k$$

subject to $x_{k+1} = A_d x_k + B_d u_k$. $Q$ penalizes error, $R$ penalizes effort, and the
ratio between them is the only real design knob.

The optimal cost-to-go turns out to be quadratic, $V(x) = x^T P x$, and $P$ satisfies the
**discrete algebraic Riccati equation**

$$P = Q + A_d^T P A_d - A_d^T P B_d \left(R + B_d^T P B_d\right)^{-1} B_d^T P A_d$$

Rather than solving that directly, iterate it. Start from $P = Q$ and apply the recursion
until $P$ stops moving - that is the finite-horizon backward pass run to convergence, which
*is* the infinite-horizon answer. The optimal feedback is then

$$K = \left(R + B_d^T P B_d\right)^{-1} B_d^T P A_d, \qquad u = -Kx$$

Use `np.linalg.solve`, never `np.linalg.inv`.

In [ ]:
def solve_dare(A, B, Q, R, tol=1e-12, max_iters=5000):
    """P, iterations = solve_dare(A, B, Q, R) by running the Riccati recursion."""
    P = np.array(Q, dtype=float)

    # DO NOT USE A WHILE LOOP
    for iteration in range(1, max_iters + 1):
        # TODO: one Riccati update. Compute the gain for the CURRENT P, then the
        #       updated P. Symmetrize with 0.5 * (P_next + P_next.T) so rounding
        #       error cannot accumulate into an asymmetric matrix.
        # TODO: return (P_next, iteration) once np.max(np.abs(P_next - P)) < tol
        raise NotImplementedError('dare not implemented')

    raise RuntimeError(f'Riccati recursion did not converge in {max_iters} iterations')


def lqr_gain(A, B, Q, R):
    """K, P, iterations. The optimal feedback is u = -K x."""
    P, iterations = solve_dare(A, B, Q, R)
    # TODO: form K from the converged P
    raise NotImplementedError('lqr gain not implemented')


def dare_residual(A, B, Q, R, P):
    """How badly P fails the Riccati equation; should be ~0 at the solution."""
    gain = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)
    return float(np.abs(Q + A.T @ P @ A - A.T @ P @ B @ gain - P).max())      # given


def simulate_linear(A, B, K, x0, steps):
    """Roll the closed-loop LINEAR model forward. Returns states and inputs."""
    X = np.zeros((steps + 1, A.shape[0]))
    U = np.zeros(steps)
    X[0] = x0
    # DO NOT USE A WHILE LOOP
    for k in range(steps):
        # TODO: apply u = -K x, store it in U[k], then step the state into X[k + 1]
        raise NotImplementedError('closed-loop simulation not implemented')
    return X, U

In [ ]:
# ------------------------------------------------------------------- TEST CELL (C)
# do not change Q, R, dt or the initial tilt in this cell
Q = np.diag([1.0, 10.0, 1.0, 1.0])     # position, PITCH, speed, pitch rate
R = np.array([[1.0]])
HORIZON = 8.0                          # the slowest closed-loop mode needs a few seconds

K, P, riccati_iterations = lqr_gain(Ad, Bd, Q, R)
assert K.shape == (1, 4)
assert np.allclose(P, P.T), 'P must be symmetric'
assert np.all(np.linalg.eigvalsh(P) > 0), 'P must be positive definite'
assert dare_residual(Ad, Bd, Q, R, P) < 1e-8, 'P does not satisfy the Riccati equation'

# Independent cross-check against a direct Schur solver.
from scipy.linalg import solve_discrete_are
np.testing.assert_allclose(P, solve_discrete_are(Ad, Bd, Q, R), rtol=1e-6, atol=1e-8)

closed_loop = np.abs(np.linalg.eigvals(Ad - Bd @ K))
assert np.all(closed_loop < 1), f'closed loop is unstable: {closed_loop}'

x0 = np.array([0.0, np.deg2rad(5), 0.0, 0.0])
steps = int(round(HORIZON / DT))
X, U = simulate_linear(Ad, Bd, K, x0, steps)
assert X.shape == (steps + 1, 4) and np.all(np.isfinite(X))
np.testing.assert_allclose(X[0], x0)
assert abs(np.rad2deg(X[-1, 1])) < 1e-2, 'pitch should come back to upright'
settling = np.argmax(np.abs(np.rad2deg(X[:, 1])) < 0.1) * DT
assert 0 < settling < 0.5, f'settling time {settling:.2f} s is out of range'

# The cost-to-go predicts the cost the optimal policy actually pays.
realized = sum(X[k] @ Q @ X[k] + U[k] * R[0, 0] * U[k] for k in range(steps))
np.testing.assert_allclose(realized, x0 @ P @ x0, rtol=1e-3)

torque_limit = model.torque_limits[2] + model.torque_limits[5]
assert np.abs(U).max() < torque_limit, 'the motors cannot deliver this torque'

print(f'Riccati converged in {riccati_iterations} iterations, residual '
      f'{dare_residual(Ad, Bd, Q, R, P):.2e}')
print(f'K = {np.round(K, 4)}')
print(f'closed-loop |eig| = {np.round(closed_loop, 6)}')
print(f'settles below 0.1 deg in {settling:.2f} s, peak torque {np.abs(U).max():.3f} Nm '
      f'of {torque_limit:.2f} available')
print(f'realized cost {realized:.4f} vs predicted x0\' P x0 = {x0 @ P @ x0:.4f}')
print('\nPart C tests passed.')

In [ ]:
# this cell is given, no TODOs here
times = np.arange(len(X)) * DT
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(9, 9))
axes[0].plot(times, np.rad2deg(X[:, 1]))
axes[0].axhline(0, color='gray', lw=0.8)
axes[0].set(ylabel='Pitch (deg)', title=f'LQR recovery from a 5 deg tilt at {TARGET_HEIGHT:.2f} m')
axes[1].plot(times, X[:, 0], label='position s (m)')
axes[1].plot(times, X[:, 2], label='speed (m/s)')
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set(ylabel='Cart states')
axes[1].legend()
axes[2].plot(times[:-1], U, label='total wheel torque')
for sign in (1, -1):
    axes[2].axhline(sign * torque_limit, color='tab:red', ls=':',
                    label='motor limit' if sign > 0 else None)
axes[2].set(xlabel='Time (s)', ylabel='Torque (Nm)')
axes[2].legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
circle = np.exp(1j * np.linspace(0, 2 * np.pi, 200))
ax.plot(circle.real, circle.imag, color='gray', lw=1, label='unit circle')
openloop, closed = np.linalg.eigvals(Ad), np.linalg.eigvals(Ad - Bd @ K)
ax.scatter(openloop.real, openloop.imag, marker='x', s=80, color='tab:red', label='open loop')
ax.scatter(closed.real, closed.imag, marker='o', s=60, color='tab:blue', label='closed loop')
ax.set(xlabel='Real', ylabel='Imaginary', title='Discrete-time poles')
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

Look at the pole plot. Open loop, one pole sits outside the unit circle - that is the robot
falling. Closed loop, feedback has dragged every pole inside. That is the entire content of
"the controller works".

Look at the torque trace too: peak demand is well under a tenth of what the motors can
deliver. This equilibrium is easy to hold. It is *reaching* it, from far away, that would be
hard - and LQR would not help you there, because the linearization stops being valid.

## Part D (5 pts BONUS): Tuning, and what we threw away

$Q$ and $R$ are the design. Larger $Q$ entries mean "I care about this error"; larger $R$
means "be gentle". Only the ratio matters - scaling both by the same factor leaves $K$
unchanged.

In [ ]:
def tuning_study(Ad, Bd, x0, weights):
    """For each pitch weight, return (settling time, peak torque, |closed-loop eigs|).

    Q = diag([1, weight, 1, 1]) and R = 1 throughout, so `weight` is exactly how much
    more we care about pitch error than about everything else.
    """
    rows = {}
    for weight in weights:
        # TODO: build Q for this weight, solve for K, simulate from x0, and record
        #       (settling_time, peak_torque, max |closed-loop eigenvalue|).
        #       settling time = first time |pitch| stays under 0.1 deg,
        #       i.e. np.argmax(np.abs(np.rad2deg(X[:, 1])) < 0.1) * DT
        raise NotImplementedError('tuning study not implemented')
    return rows

In [ ]:
# ------------------------------------------------------------------- TEST CELL (D)
weights = [0.1, 1.0, 10.0, 100.0, 1000.0]
rows = tuning_study(Ad, Bd, x0, weights)
assert set(rows) == set(weights), 'one row per weight, please'

print(f'{"pitch weight":>13}{"settling (s)":>14}{"peak torque (Nm)":>19}{"max |eig|":>12}')
for weight in weights:
    settling_w, torque_w, radius_w = rows[weight]
    print(f'{weight:>13g}{settling_w:>14.2f}{torque_w:>19.3f}{radius_w:>12.6f}')
    assert radius_w < 1, f'weight {weight} gives an unstable closed loop'

# Caring more about pitch buys a faster recovery, and pays for it in torque.
assert rows[1000.0][0] <= rows[0.1][0], 'a heavier pitch weight should not be slower'
assert rows[1000.0][1] > rows[0.1][1], 'a heavier pitch weight should cost more torque'
print('\nPart D tests passed.')

### Short answer

**1.** Scaling both $Q$ and $R$ by 10 leaves $K$ unchanged. Why?

**PUT YOUR ANSWER HERE**

**2.** The table shows faster recovery costing more torque. On the real robot, what
eventually stops you from simply pushing the pitch weight higher?

**PUT YOUR ANSWER HERE**

**3.** Name two things this planar model throws away that a real pineapple_v0 has, and say
when each would matter.

**PUT YOUR ANSWER HERE**

**4.** We linearized about one equilibrium height. What would you expect to happen if you
used this same $K$ with the legs extended to a much taller stance?

**PUT YOUR ANSWER HERE**

### Submitting

1. **Kernel -> Restart Kernel and Run All Cells.**
2. Check that every test cell printed its `... tests passed.` line.
3. Check that you answered all four short-answer questions in Part D.
4. Export to PDF: `File -> Save and Export Notebook As -> HTML`, then print to PDF.
5. Submit the PDF **and** the `.ipynb`.